# Carbon Emission Analysis and Environmental Factors Using Python

This project analyzes global sustainable-energy data to understand carbon-emission patterns and relationships with energy and economic indicators.

**Tools:** Python, Pandas, NumPy, Matplotlib, Seaborn, Scikit-learn


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

df = pd.read_csv("../data/global_data_on_sustainable_energy.csv")
df.head()


## 1. Dataset Overview

In [ ]:
print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nData types:")
display(df.dtypes)


## 2. Data Cleaning and Quality Checks

In [ ]:
# Remove accidental whitespace from column names
df.columns = [c.strip() for c in df.columns]

print("Missing values:")
display(df.isnull().sum().sort_values(ascending=False).head(15))

print("Duplicate rows:", df.duplicated().sum())

# Convert key fields to numeric
df["Year"] = pd.to_numeric(df["Year"], errors="coerce")
df["Value_co2_emissions_kt_by_country"] = pd.to_numeric(
    df["Value_co2_emissions_kt_by_country"], errors="coerce"
)

df = df.drop_duplicates()


## 3. Global CO₂ Emission Trend

In [ ]:
co2_col = "Value_co2_emissions_kt_by_country"
global_trend = df.groupby("Year")[co2_col].sum()

plt.figure(figsize=(10, 6))
plt.plot(global_trend.index, global_trend.values, marker="o")
plt.title("Global CO₂ Emissions Trend")
plt.xlabel("Year")
plt.ylabel("CO₂ Emissions (kt)")
plt.grid(alpha=0.2)
plt.show()


## 4. Top 10 Countries by Total CO₂ Emissions

In [ ]:
top10 = (
    df.groupby("Entity")[co2_col]
      .sum()
      .sort_values(ascending=False)
      .head(10)
      .sort_values()
)

plt.figure(figsize=(10, 6))
plt.barh(top10.index, top10.values)
plt.title("Top 10 Countries by Total CO₂ Emissions")
plt.xlabel("CO₂ Emissions (kt)")
plt.ylabel("Country")
plt.show()


## 5. India Trend

In [ ]:
india = df[df["Entity"].eq("India")].sort_values("Year")

plt.figure(figsize=(10, 6))
plt.plot(india["Year"], india[co2_col], marker="o")
plt.title("India CO₂ Emissions Trend")
plt.xlabel("Year")
plt.ylabel("CO₂ Emissions (kt)")
plt.show()


## 6. India vs China vs United States

In [ ]:
selected = df[df["Entity"].isin(["India", "China", "United States"])]

plt.figure(figsize=(10, 6))
for country, group in selected.groupby("Entity"):
    group = group.sort_values("Year")
    plt.plot(group["Year"], group[co2_col], marker="o", label=country)

plt.title("CO₂ Emissions: India vs China vs United States")
plt.xlabel("Year")
plt.ylabel("CO₂ Emissions (kt)")
plt.legend()
plt.show()


## 7. Correlation Analysis

In [ ]:
candidate = [
    "Value_co2_emissions_kt_by_country",
    "Renewable energy share in the total final energy consumption (%)",
    "Electricity from fossil fuels (TWh)",
    "Electricity from renewables (TWh)",
    "Primary energy consumption per capita (kWh/person)",
    "Energy intensity level of primary energy (MJ/$2021 PPP GDP)",
    "GDP per capita",
    "gdp_growth"
]
heat_cols = [c for c in candidate if c in df.columns]

corr = df[heat_cols].corr(numeric_only=True)

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation Heatmap of Key Variables")
plt.show()


## 8. Multiple Linear Regression

In [ ]:
features = [
    "Electricity from fossil fuels (TWh)",
    "Electricity from renewables (TWh)",
    "Primary energy consumption per capita (kWh/person)",
    "Energy intensity level of primary energy (MJ/$2021 PPP GDP)",
    "GDP per capita",
    "gdp_growth"
]
features = [c for c in features if c in df.columns]

model_df = df[[co2_col] + features].dropna()

X = model_df[features]
y = model_df[co2_col]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42
)

model = LinearRegression()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print("Features:", features)
print(f"MAE: {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R²: {r2:.4f}")


## 9. Actual vs Predicted CO₂ Emissions

In [ ]:
plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.6)

mn = min(y_test.min(), y_pred.min())
mx = max(y_test.max(), y_pred.max())
plt.plot([mn, mx], [mn, mx], linestyle="--")

plt.title("Actual vs Predicted CO₂ Emissions")
plt.xlabel("Actual CO₂ Emissions (kt)")
plt.ylabel("Predicted CO₂ Emissions (kt)")
plt.show()


## 10. Conclusion

The analysis examines CO₂-emission patterns across countries and years and explores their relationships with energy and economic indicators. The regression section provides quantitative model evaluation using MAE, RMSE, and R².

**Important:** Conclusions should be based on the numerical results produced by running the notebook rather than assumed in advance.
